### This notebook should be run with the following kernel and on wICE

`/data/leuven/software/biomed/jupyter_kernels/cbd_bioinf_v4/`

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scarches.dataset.trvae.data_handling import remove_sparsity
import torch
import scvi

import anndata
import scvi
import scanpy as sc

In [ ]:
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

scvi.settings.seed = 94705


%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

In [ ]:
model_dir_path = '/scANVI_Models/Public_Data/Allen_Brain_Atlas/M1/cluster_label' # change path as necessary

In [ ]:
query_adata = sc.read_h5ad(f'./matrices/20250209_all_PB-002_cg.h5ad')

In [ ]:
query_adata

In [ ]:
query_adata.obs

In [ ]:
# Add batch column
query_adata.obs['external_donor_name_label'] = query_adata.obs['sample'].values

In [ ]:
import os

In [ ]:
# Find any procceses locking the GPU, and if it's not this one, kill it - Use carefully on multi GPU jobs
try:
    nvidia_pid = ! nvidia-smi -q | grep 'Process ID' 
    nvidia_pid = nvidia_pid[0].split(':')[1].strip()

    if os.getpid() != int(nvidia_pid):
        print(f'Killing: {nvidia_pid}')
        ! kill {nvidia_pid}
except:
    print("GPU wasn't locked")
    pass

In [ ]:
if "PCs" in query_adata.varm.keys():
    del query_adata.varm["PCs"]

In [ ]:
scvi.model.SCVI.prepare_query_anndata(
    query_adata,
    model_dir_path
)

In [ ]:
query_adata.layers['counts'] = query_adata.raw.X if query_adata.raw else query_adata.X

In [ ]:
vae_q = scvi.model.SCANVI.load_query_data(
    query_adata,
    model_dir_path,
)

In [ ]:
vae_q.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [ ]:
query_adata.obsm["X_scANVI"] = vae_q.get_latent_representation()
query_adata.obs["predictions"] = vae_q.predict()


In [ ]:
! mkdir -p scANVI

In [ ]:
import datetime

In [ ]:
index_data = query_adata.obs.index
column_data = query_adata.obs['predictions']

# Create a DataFrame with the index and column data
df = pd.DataFrame({'index': index_data, 'column_name': column_data})

date = datetime.datetime.now().strftime('%Y%m%d')
# Write the DataFrame to a CSV file
df.to_csv(f'scANVI/{date}_PB-002_celltype_from_scANVI_cg.csv', index=False)

# Subclass label model

In [ ]:
model_dir_path = '/scANVI_Models/Public_Data/Allen_Brain_Atlas/M1/subclass_label' #change path as necessary

In [ ]:
query_adata = sc.read_h5ad(f'./matrices/20250209_all_PB-002_cg.h5ad')

In [ ]:
# Add batch column

query_adata.obs['external_donor_name_label'] = query_adata.obs['sample'].values

In [ ]:
if "PCs" in query_adata.varm.keys():
    del query_adata.varm["PCs"]

In [ ]:
scvi.model.SCVI.prepare_query_anndata(
    query_adata,
    model_dir_path
)

In [ ]:
query_adata.layers['counts'] = query_adata.raw.X if query_adata.raw else query_adata.X

In [ ]:
vae_q = scvi.model.SCANVI.load_query_data(
    query_adata,
    model_dir_path,
)

In [ ]:
vae_q.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [ ]:
query_adata.obsm["X_scANVI"] = vae_q.get_latent_representation()
query_adata.obs["predictions"] = vae_q.predict()


In [ ]:
index_data = query_adata.obs.index
column_data = query_adata.obs['predictions']

# Create a DataFrame with the index and column data
df = pd.DataFrame({'index': index_data, 'column_name': column_data})


# Write the DataFrame to a CSV file
df.to_csv(f'scANVI/{date}_PB-002_subclusters_from_scANVI_cg.csv', index=False)

In [ ]:
pip list